# LLM Evaluation
Measuring the quality of generated text.

In [ ]:
import micropip
await micropip.install(['scikit-learn','matplotlib','numpy'])
print('Ready!')

## Why It Is Hard

- Multiple valid answers exist
- Fluency != factual accuracy
- Context-dependent quality
- Human labels are expensive

**Solution:** BLEU/ROUGE for automatic + LLM-as-judge for quality.

In [ ]:
# BLEU-1 (precision-based)
from collections import Counter
def bleu1(reference, hypothesis):
    ref = reference.lower().split()
    hyp = hypothesis.lower().split()
    ref_c = Counter(ref)
    hyp_c = Counter(hyp)
    clipped = {w: min(cnt, ref_c.get(w,0)) for w,cnt in hyp_c.items()}
    prec = sum(clipped.values()) / max(len(hyp),1)
    bp = min(1.0, len(hyp)/max(len(ref),1))
    return bp * prec
ref = 'Neural networks learn to classify images by training on labelled data'
hyps = [
    'Neural networks learn to classify images by training on labelled data',
    'Neural networks learn image classification through training',
    'Deep learning classifies pictures',
    'The weather today is sunny and warm',
]
print('BLEU-1 scores:')
for h in hyps:
    s = bleu1(ref,h)
    print(f'  {s:.3f} {"#"*int(s*25)} | {h[:55]}...')

In [ ]:
# ROUGE-1 F1
def rouge1(ref, hyp):
    ref_s = set(ref.lower().split())
    hyp_l = hyp.lower().split()
    hyp_s = set(hyp_l)
    overlap = len(ref_s & hyp_s)
    p = overlap / max(len(hyp_l),1)
    r = overlap / max(len(ref_s),1)
    f1 = 2*p*r / max(p+r,1e-8)
    return p,r,f1
ref = 'Machine learning algorithms learn patterns from training data to make predictions'
summaries = [
    'ML algorithms learn from training data to make predictions',
    'Algorithms learn from data',
    'Machine learning deep learning neural network data science model',
]
print(f'{'Summary':<55}| P     | R     | F1')
print('-'*75)
for s in summaries:
    p,r,f1 = rouge1(ref,s)
    print(f'{s[:55]:<55}| {p:.3f} | {r:.3f} | {f1:.3f}')

## LLM-as-Judge Prompt Pattern

```
You are an expert evaluator. Rate on 1-5:
1. Faithfulness: Is it factually grounded?
2. Relevance: Does it answer the question?
3. Completeness: Are key points covered?
4. Clarity: Is it well-written?

Respond in JSON: {faithfulness: X, relevance: X, completeness: X, clarity: X, overall: X}
```

Cost: ~$0.0001 per evaluation with GPT-4o-mini. Automated at scale.

## RAGAS -- Automated RAG Evaluation

```python
# pip install ragas
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from datasets import Dataset

eval_data = Dataset.from_dict({
    'question':    ['What is Ridge regression?'],
    'answer':      ['Ridge adds L2 penalty to prevent overfitting.'],
    'contexts':    [['Ridge regression adds alpha*sum(w^2) to MSE loss.']],
    'ground_truth':['Ridge regression adds L2 regularisation to linear regression.']
})
result = evaluate(eval_data, metrics=[faithfulness, answer_relevancy,
                                       context_precision, context_recall])
print(result)
# {'faithfulness': 0.92, 'answer_relevancy': 0.88, ...}
```

---
**Your turn:** Write 3 different answers to 'What is dropout?' at varying quality. Compute ROUGE-1 F1 for each.